<a href="https://colab.research.google.com/github/silvia-dev-prog/ai-agents-for-beginners/blob/main/Aula25_series_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predição de Séries Temporais usando LSTM

**Francisco Aparecido Rodrigues**  
Universidade de São Paulo, São Carlos, Brasil  
<https://sites.icmc.usp.br/francisco>  

<hr>

Neste notebook vamos construir um modelo de **rede neural recorrente LSTM** (*Long Short-Term Memory*) para prever o número mensal de passageiros de uma companhia aérea, usando a biblioteca **Keras**.

Vamos usar o clássico dataset **AirPassengers**: o número total mensal de passageiros de voos internacionais entre **janeiro de 1949 e dezembro de 1960** (144 observações, em milhares de passageiros). Esse conjunto de dados tem duas características bem visíveis:

- **Tendência**: o número de passageiros cresce ao longo dos anos.
- **Sazonalidade**: existem picos e vales que se repetem todo ano (por exemplo, mais viagens em determinados meses).

O objetivo é treinar uma LSTM que aprenda o padrão da série e consiga prever valores futuros a partir de valores passados.


## 1. Importando as bibliotecas

Vamos usar:

- **numpy** e **pandas**: manipulação de dados numéricos e tabulares;
- **matplotlib**: visualização dos dados e dos resultados;
- **scikit-learn**: normalização dos dados (`MinMaxScaler`) e cálculo de erro (`mean_squared_error`);
- **keras**: construção e treinamento da rede neural LSTM.

Também fixamos as *seeds* (sementes) de aleatoriedade do NumPy e do Keras/TensorFlow. Redes neurais têm inicialização aleatória de pesos, então fixar a semente ajuda a tornar os resultados mais **reprodutíveis** (você deve obter valores parecidos com os deste notebook ao executá-lo novamente).


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, Input

# Reprodutibilidade
np.random.seed(42)
keras.utils.set_random_seed(42)

print("Versão do Keras:", keras.__version__)


Versão do Keras: 3.13.2


## 2. Carregando os dados

Para que este notebook funcione sem depender de download externo, os 144 valores mensais do dataset **AirPassengers** (1949–1960) estão embutidos diretamente no código abaixo. Cada valor representa o total de passageiros (em milhares) transportados naquele mês.

Criamos um `DataFrame` do pandas com um índice de datas mensais (`DatetimeIndex`), o que é uma boa prática ao trabalhar com séries temporais: facilita plotagem, reamostragem e interpretação dos resultados.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Caminho para o arquivo CSV no Google Drive
csv_path = '/content/drive/MyDrive/data/airline-passengers.csv'

# Carregar o dataset
df = pd.read_csv(
    csv_path,
    parse_dates=['Month'],
    index_col='Month'
)

# Renomear a coluna para consistência com o notebook
df.rename(columns={'Passengers': 'passageiros'}, inplace=True)

print("Formato dos dados:", df.shape)
df.head(12)

MessageError: Error: credential propagation was unsuccessful

## 3. Visualização

Antes de treinar qualquer modelo, é essencial visualizar os dados. Isso ajuda a identificar tendência, sazonalidade, outliers e a escala dos valores, informações que vão guiar as escolhas de pré-processamento e arquitetura do modelo.


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df.index, df["passageiros"], color="steelblue")
plt.title("Número mensal de passageiros de voos internacionais (1949-1960)")
plt.xlabel("Ano")
plt.ylabel("Passageiros (milhares)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**O que observamos no gráfico:**

- Uma **tendência de crescimento** clara ao longo dos 12 anos.
- Um padrão **sazonal** que se repete todo ano (picos em meses de mais viagens, vales em outros).
- A **amplitude da sazonalidade aumenta** com o tempo (os picos e vales ficam mais pronunciados conforme o volume total cresce), chamamos isso de sazonalidade **multiplicativa**.

Essas características tornam a série um bom desafio para uma LSTM: o modelo precisa aprender tanto a tendência de longo prazo quanto o padrão cíclico de curto prazo.


## 4. Pré-processamento dos dados

Redes neurais (LSTM inclusive) funcionam melhor quando os dados de entrada estão em uma escala pequena e consistente, geralmente entre 0 e 1. Vamos usar o `MinMaxScaler` do scikit-learn para isso.

### 4.1 Separação treino/teste

Diferente de problemas "tabulares" comuns, em séries temporais **não podemos embaralhar os dados** antes de dividir treino e teste, isso quebraria a ordem cronológica e "vazaria" informação do futuro para o passado (*data leakage*). Por isso, separamos os primeiros ~67% dos meses para treino e os últimos ~33% para teste, mantendo a ordem temporal.

Também é importante **ajustar (`fit`) o normalizador apenas com os dados de treino**, e depois aplicá-lo (`transform`) tanto no treino quanto no teste. Se normalizássemos com toda a série, o modelo teria acesso indireto a informações do futuro (do conjunto de teste) durante o treinamento.


In [ ]:
valores = df["passageiros"].values.reshape(-1, 1).astype("float32")

tamanho_treino = int(len(valores) * 0.67)
tamanho_teste = len(valores) - tamanho_treino

dados_treino_bruto = valores[:tamanho_treino]
dados_teste_bruto = valores[tamanho_treino:]

print(f"Total de meses: {len(valores)}")
print(f"Meses de treino: {len(dados_treino_bruto)}")
print(f"Meses de teste:  {len(dados_teste_bruto)}")

scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(dados_treino_bruto)  # ajusta (fit) somente com dados de treino

dados_treino = scaler.transform(dados_treino_bruto)
dados_teste = scaler.transform(dados_teste_bruto)


### 4.2 Criando janelas deslizantes (*sliding windows*)

Uma LSTM aprende com exemplos no formato "dado o passado recente, preveja o próximo valor". Para transformar nossa série (uma sequência de números) nesse formato de aprendizado supervisionado, usamos uma **janela deslizante**: cada exemplo de entrada (`X`) é composto pelos últimos `look_back` meses, e o alvo (`y`) é o mês seguinte.

Por exemplo, com `look_back = 12` (um ano de histórico):

```
X[0] = [mês 1, mês 2, ..., mês 12]  ->  y[0] = mês 13
X[1] = [mês 2, mês 3, ..., mês 13]  ->  y[1] = mês 14
...
```

Escolhemos `look_back = 12` porque a sazonalidade da série é anual, assim o modelo tem acesso a um ciclo completo para tentar capturar o padrão.

In [ ]:
def criar_janelas(dados, look_back=12):
    '''Transforma uma série 1D em pares (X, y) usando janela deslizante.'''
    X, y = [], []
    for i in range(len(dados) - look_back):
        X.append(dados[i:i + look_back, 0])
        y.append(dados[i + look_back, 0])
    return np.array(X), np.array(y)

look_back = 12

X_treino, y_treino = criar_janelas(dados_treino, look_back)
X_teste, y_teste = criar_janelas(dados_teste, look_back)

print("Formato de X_treino:", X_treino.shape)
print("Formato de y_treino:", y_treino.shape)
print("Formato de X_teste:", X_teste.shape)
print("Formato de y_teste:", y_teste.shape)


### 4.3 Ajustando o formato para a LSTM

Camadas LSTM do Keras esperam entradas com 3 dimensões: `(amostras, passos_de_tempo, features)`.

- **amostras**: quantos exemplos temos;
- **passos_de_tempo** (*timesteps*): quantos meses de histórico cada exemplo contém (`look_back`);
- **features**: quantas variáveis medimos em cada passo de tempo (aqui, só 1: o número de passageiros).

Nosso `X` está no formato `(amostras, look_back)`, então precisamos adicionar a dimensão de *features*.


In [ ]:
X_treino = X_treino.reshape((X_treino.shape[0], X_treino.shape[1], 1))
X_teste = X_teste.reshape((X_teste.shape[0], X_teste.shape[1], 1))

print("Formato final de X_treino:", X_treino.shape)
print("Formato final de X_teste:", X_teste.shape)

## 5. Construindo o modelo LSTM

Vamos usar a API `Sequential` do Keras, empilhando camadas em ordem:

1. **`Input`**: define o formato de entrada `(look_back, 1)`.
2. **`LSTM(50)`**: a camada recorrente propriamente dita, com 50 unidades (neurônios). É esse número de unidades que define a capacidade do modelo de armazenar informação sobre o passado.
3. **`Dropout(0.2)`**: técnica de regularização que "desliga" aleatoriamente 20% das conexões durante o treino, ajudando a evitar *overfitting* (quando o modelo decora o treino em vez de generalizar).
4. **`Dense(1)`**: camada de saída totalmente conectada com 1 neurônio, que produz a previsão de um único valor (o próximo mês).

Compilamos o modelo com:

- **Otimizador `adam`**: método padrão e eficiente para ajustar os pesos da rede;
- **Função de perda `mean_squared_error` (MSE)**: adequada para problemas de regressão como este, penalizando mais fortemente erros grandes.


In [ ]:
model = Sequential([
    Input(shape=(look_back, 1)),
    LSTM(50, activation="tanh"),
    Dropout(0.2),
    Dense(1),
])

model.compile(optimizer="adam", loss="mean_squared_error")
model.summary()

## 6. Treinando o modelo

Alguns conceitos importantes do treinamento:

- **Época (`epoch`)**: uma passada completa pelos dados de treino;
- **Tamanho do lote (`batch_size`)**: quantos exemplos o modelo processa antes de atualizar os pesos uma vez;
- **`validation_data`**: usamos o conjunto de teste aqui apenas para *acompanhar* a perda em dados não vistos durante o treino — isso nos ajuda a perceber overfitting (quando a perda de treino cai mas a de validação sobe ou estagna).

Como a série é pequena (só ~144 pontos), o treinamento é rápido mesmo com bastante épocas.


In [ ]:
historico = model.fit(
    X_treino, y_treino,
    validation_data=(X_teste, y_teste),
    epochs=200,
    batch_size=8,
    verbose=0,
)

print("Treinamento concluído.")
print(f"Perda final (treino): {historico.history['loss'][-1]:.5f}")
print(f"Perda final (validação): {historico.history['val_loss'][-1]:.5f}")


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(historico.history["loss"], label="Perda (treino)")
plt.plot(historico.history["val_loss"], label="Perda (validação/teste)")
plt.title("Evolução da função de perda (MSE) durante o treinamento")
plt.xlabel("Época")
plt.ylabel("MSE (na escala normalizada)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


Se a curva de validação parar de cair (ou começar a subir) enquanto a de treino continua caindo, é sinal de **overfitting** — nesse caso poderíamos usar `EarlyStopping`, mais `Dropout`, menos épocas ou mais dados. Observe o comportamento das duas curvas acima antes de seguir para a avaliação.


## 7. Fazendo previsões e avaliando o modelo

Agora vamos usar o modelo treinado para prever os valores de treino e teste, e depois **desfazer a normalização** (`inverse_transform`) para voltar à escala original (número real de passageiros), o que torna os resultados interpretáveis e permite calcular o erro em unidades reais (milhares de passageiros).

Usaremos o **RMSE** (raiz do erro quadrático médio) como métrica: ele tem a mesma unidade da variável original, o que facilita a interpretação ("em média, erramos por X mil passageiros").


In [ ]:
previsao_treino = model.predict(X_treino, verbose=0)
previsao_teste = model.predict(X_teste, verbose=0)

# Desfazendo a normalização (voltando à escala original)
previsao_treino = scaler.inverse_transform(previsao_treino)
y_treino_real = scaler.inverse_transform(y_treino.reshape(-1, 1))

previsao_teste = scaler.inverse_transform(previsao_teste)
y_teste_real = scaler.inverse_transform(y_teste.reshape(-1, 1))

rmse_treino = np.sqrt(mean_squared_error(y_treino_real, previsao_treino))
rmse_teste = np.sqrt(mean_squared_error(y_teste_real, previsao_teste))

print(f"RMSE (treino): {rmse_treino:.2f} mil passageiros")
print(f"RMSE (teste):  {rmse_teste:.2f} mil passageiros")


### Visualizando previsões vs. valores reais

Para visualizar como as previsões se encaixam na série original, vamos construir dois vetores do mesmo tamanho da série completa — um para as previsões de treino e outro para as de teste — preenchidos com `NaN` fora do intervalo correspondente. Assim conseguimos sobrepor as três curvas (real, previsto no treino, previsto no teste) no mesmo gráfico, respeitando o deslocamento causado pelo `look_back`.


In [ ]:
serie_completa = valores  # escala original, formato (n, 1)

plot_treino = np.full_like(serie_completa, np.nan, dtype="float64")
plot_treino[look_back:len(previsao_treino) + look_back] = previsao_treino

plot_teste = np.full_like(serie_completa, np.nan, dtype="float64")
inicio_teste = len(previsao_treino) + (look_back * 2)
plot_teste[inicio_teste:inicio_teste + len(previsao_teste)] = previsao_teste

plt.figure(figsize=(13, 5))
plt.plot(df.index, serie_completa, label="Real", color="black", linewidth=1.5)
plt.plot(df.index, plot_treino, label="Previsto (treino)", color="tab:blue")
plt.plot(df.index, plot_teste, label="Previsto (teste)", color="tab:orange")
plt.title("Passageiros reais vs. previstos pela LSTM")
plt.xlabel("Ano")
plt.ylabel("Passageiros (milhares)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


Repare que a curva prevista para o **teste** (laranja) é a parte mais importante para avaliar o modelo: são meses que a LSTM nunca viu durante o treinamento. Se ela acompanha razoavelmente bem a tendência e a sazonalidade da curva real (preta), o modelo aprendeu um padrão útil — mesmo que não acerte cada mês perfeitamente.


## 8. Prevendo meses futuros (fora da amostra)

Além de avaliar no conjunto de teste, podemos usar o modelo para **projetar meses além do fim da série** (dezembro de 1960 em diante). Fazemos isso de forma **recursiva** (também chamada de previsão *multi-step*):

1. Começamos com os últimos `look_back` meses conhecidos;
2. Prevemos o próximo mês;
3. Adicionamos essa previsão ao final da janela e removemos o valor mais antigo;
4. Repetimos o processo quantas vezes quisermos prever à frente.

**Atenção:** como cada previsão passa a ser usada como entrada para a próxima, o erro tende a **acumular** conforme avançamos no horizonte — por isso previsões recursivas de longo prazo devem ser vistas com cautela.


In [ ]:
def prever_futuro(modelo, ultima_janela, n_passos, scaler):
    '''Gera previsões recursivas para n_passos meses à frente.'''
    janela_atual = ultima_janela.copy()  # formato (look_back, 1), escala normalizada
    previsoes_normalizadas = []

    for _ in range(n_passos):
        entrada = janela_atual.reshape((1, janela_atual.shape[0], 1))
        proximo_valor = modelo.predict(entrada, verbose=0)[0, 0]
        previsoes_normalizadas.append(proximo_valor)

        # desliza a janela: remove o mais antigo, adiciona a nova previsão
        janela_atual = np.append(janela_atual[1:], [[proximo_valor]], axis=0)

    previsoes = scaler.inverse_transform(np.array(previsoes_normalizadas).reshape(-1, 1))
    return previsoes

n_meses_futuro = 12
ultima_janela = dados_teste[-look_back:]  # últimos 12 meses conhecidos (normalizados)

previsoes_futuras = prever_futuro(model, ultima_janela, n_meses_futuro, scaler)

datas_futuras = pd.date_range(
    start=df.index[-1] + pd.DateOffset(months=1),
    periods=n_meses_futuro,
    freq="MS",
)

df_futuro = pd.DataFrame({"passageiros_previstos": previsoes_futuras.flatten()}, index=datas_futuras)
df_futuro


In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(df.index, df["passageiros"], label="Histórico real", color="black")
plt.plot(df_futuro.index, df_futuro["passageiros_previstos"],
         label="Previsão futura (12 meses)", color="crimson", linestyle="--", marker="o")
plt.axvline(df.index[-1], color="gray", linestyle=":", linewidth=1)
plt.title("Previsão recursiva para os 12 meses seguintes")
plt.xlabel("Ano")
plt.ylabel("Passageiros (milhares)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Conclusão

Neste notebook, percorremos o fluxo completo de um projeto de previsão de séries temporais com redes neurais:

1. Carregamos e visualizamos a série de passageiros, identificando tendência e sazonalidade;
2. Normalizamos os dados e organizamos a série em janelas deslizantes para aprendizado supervisionado;
3. Construímos e treinamos uma LSTM simples com Keras;
4. Avaliamos o modelo com RMSE e visualização das previsões no conjunto de teste;
5. Geramos uma previsão recursiva para meses futuros.

**Limitações**

- O dataset é **pequeno** (144 pontos): modelos de deep learning costumam se beneficiar de muito mais dados;
- A previsão recursiva **acumula erro** ao longo do horizonte; quanto mais longe no futuro, menos confiável ela tende a ser;
- Não fizemos uma busca sistemática de **hiperparâmetros** (número de unidades LSTM, `look_back`, `batch_size`, número de épocas, etc.);

### Exercícios de fixação

1 -  Empilhar mais de uma camada LSTM (`return_sequences=True` na primeira camada).

2 - Testar diferentes tamanhos de janela (`look_back`).